# CARec encoder and dimension study

- Load the saved Amazon Electronics catalogue, interaction partitions, and ID maps.
- Use the `NI-PML benchmarks` kernel from the project `.venv`.
- Data exploration, text preparation, sampling, splitting, and saving belong in [data_exploration.ipynb](data_exploration.ipynb).


In [1]:
from pathlib import Path
import gzip
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


## Load products and ID maps

- Read one product record per JSONL line.
- Load user and product ID maps, keeping their IDs as strings.


In [2]:
items_path = PREPARED_DIR / "items.jsonl.gz"
product_records = []

with gzip.open(items_path, "rt", encoding="utf-8") as source:
    for line in source:
        product = json.loads(line)
        product_records.append(product)

items = pd.DataFrame(product_records)
del product_records

user_map_path = PREPARED_DIR / "user_ids.tsv.gz"
user_map = pd.read_csv(user_map_path, sep="\t", dtype={"user_id": str})

item_map_path = PREPARED_DIR / "item_ids.tsv.gz"
item_map = pd.read_csv(item_map_path, sep="\t", dtype={"item_id": str})


## Load interaction partitions

- Load train, validation, test, and cold interactions.
- Keep user and product IDs as strings and preserve float precision.


In [3]:
partition_names = ("train", "valid", "test", "cold")

interaction_types = {
    "user_id:token": str,
    "item_id:token": str,
}
partitions = {}

for name in partition_names:
    partition_path = PREPARED_DIR / f"{name}.tsv.gz"
    frame = pd.read_csv(
        partition_path,
        sep="\t",
        dtype=interaction_types,
        float_precision="round_trip",
    )
    partitions[name] = frame

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


## Show the loaded data

- Print product, user, and partition counts.
- Show two product texts.


In [4]:
print(f"Loaded {len(items):,} products and {len(user_map):,} users")

for name, frame in partitions.items():
    print(f"{name}: {len(frame):,} interactions")

text_preview = items[["item_id:token", "text"]].head(2)
display(text_preview)


Loaded 34,220 products and 81,520 users
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...


## Encoder candidates

- `text` is the saved product text from `items["text"]`; `instruction` describes the recommendation task. The pilot instruction below is provisional.
- Run the three instruction models with and without instructions: plain `text` for Qwen/E5-instruct, an empty prompt for INSTRUCTOR. Pilot these variants before bulk generation.
- E5-base's `query: ` prefix is required formatting, not a task instruction. `\n` below means a newline.

| Model | Instruction-based | Input format |
| --- | --- | --- |
| [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | No | `text` |
| [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | No | `text` |
| [e5-base-v2](https://huggingface.co/intfloat/e5-base-v2) | No | `query: {text}` |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | Yes | `Instruct: {instruction}\nQuery:{text}` |
| [multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct) | Yes | `Instruct: {instruction}\nQuery: {text}` |
| [instructor-base](https://huggingface.co/hkunlp/instructor-base) | Yes | `text` with `prompt=instruction`, excluding the prompt from pooling |


## Full-catalogue embeddings

- Encode the same saved catalogue for all six models, keeping native dimensions and normalized float32 vectors.
- Save nine NPZ files, including both modes for each instruction model. Each file contains ordered product IDs and encoder settings.
- Use batches of 32, or 16 for Qwen. The generation run loads one model per process.
- Keep the shared instruction unchanged. Preview cells still show a single product.
- Load existing condition files before loading model weights. Generate only missing files; mismatched IDs or settings raise an error.


In [5]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

PILOT_INSTRUCTION = "Encode this sentence for reccomendation purposes"

device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 4

print("running on", device)
print()
sample_texts = items["text"].head(batch_size).tolist()
print("Sample text:", sample_texts[0])

/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


running on mps

Sample text: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [6]:
def save_embeddings(path, item_ids, embeddings, model, model_id, prompt="", normalized=True):
    """Save embeddings, ordered product IDs, and encoder settings to one NPZ file."""
    path = Path(path)
    item_ids = np.asarray(item_ids, dtype=str)
    if embeddings.ndim != 2 or len(item_ids) != len(embeddings):
        raise ValueError("Product IDs and embedding rows must match.")

    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("wb") as output:
        np.savez(
            output,
            item_ids=item_ids,
            embeddings=embeddings,
            model_id=model_id,
            revision=model[0].auto_model.config._commit_hash,
            prompt=prompt,
            normalized=normalized,
            max_tokens=model.max_seq_length,
        )
    temporary_path.replace(path)
    print(f"Saved {len(item_ids):,} products to {path}")
    return path


In [7]:
def load_embeddings(path, item_ids, model_id, prompt, dimension):
    """Load a saved condition and check its IDs and encoder settings."""
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved["item_ids"], np.asarray(item_ids, dtype=str)):
            raise ValueError(f"Product IDs do not match: {path}")
        if (str(saved["model_id"]) != model_id
                or str(saved["prompt"]) != prompt
                or not bool(saved["normalized"])):
            raise ValueError(f"Encoder settings do not match: {path}")
        vectors = saved["embeddings"]
    if vectors.shape != (len(item_ids), dimension) or vectors.dtype != np.float32:
        raise ValueError(f"Embedding shape or dtype does not match: {path}")
    if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4):
        raise ValueError(f"Invalid embedding values: {path}")
    print(f"Loaded {len(item_ids):,} products from {path}")
    return vectors


In [8]:
catalogue_texts = items["text"].tolist()
catalogue_item_ids = items["item_id:token"].to_numpy(dtype=str)
embedding_dir = PROJECT_ROOT / "artifacts/embeddings" / PREPARED_DIR.name
full_batch_size = 32
print(f"Catalogue: {len(catalogue_texts):,} products")


Catalogue: 34,220 products


### MiniLM-L6-v2

- 22.7 million parameters, 6 Transformer layers.
- Output dimension: 384.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 256 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

- Encode the full saved catalogue, including cold products, in batches of 32. Keep normalized 384-dimensional vectors.
- Save vectors and product IDs in the same row order. Run the save cell after encoding. Existing files are reused; interrupted, unsaved conditions must restart.


In [9]:
minilm_model_id = "sentence-transformers/all-MiniLM-L6-v2"
minilm_batch_size = 32
minilm_texts = items["text"].tolist()
minilm_item_ids = items["item_id:token"].to_numpy(dtype=str)
minilm_output_path = embedding_dir / "all-MiniLM-L6-v2.npz"

if minilm_output_path.exists():
    minilm_embeddings = load_embeddings(
        minilm_output_path, minilm_item_ids, minilm_model_id, '', 384,
    )
else:
    minilm = SentenceTransformer(
        minilm_model_id,
        device=device,
        cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
    )
    minilm_embeddings = minilm.encode(
        minilm_texts,
        batch_size=minilm_batch_size,
        prompt='',
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    assert minilm_embeddings.shape == (len(minilm_item_ids), 384)
    assert np.isfinite(minilm_embeddings).all(), "Non-finite embeddings"
print(minilm_embeddings.shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/all-MiniLM-L6-v2.npz
(34220, 384)


In [10]:
if not minilm_output_path.exists():
    save_embeddings(
        minilm_output_path, minilm_item_ids, minilm_embeddings,
        minilm, minilm_model_id,
    )
else:
    print(f"Already saved: {minilm_output_path}")


Already saved: /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/all-MiniLM-L6-v2.npz


### MPNet-base-v2

- 109.5 million parameters, 12 Transformer layers.
- Output dimension: 768.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 384 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-mpnet-base-v2).


In [11]:
mpnet_model_id = "sentence-transformers/all-mpnet-base-v2"
mpnet_output_path = embedding_dir / "all-mpnet-base-v2.npz"

if mpnet_output_path.exists():
    mpnet_embeddings = load_embeddings(
        mpnet_output_path, catalogue_item_ids, mpnet_model_id, '', 768,
    )
else:
    mpnet = SentenceTransformer(
        mpnet_model_id,
        device=device,
        cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
    )
    mpnet_embeddings = mpnet.encode(
        catalogue_texts,
        batch_size=full_batch_size,
        prompt='',
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    assert mpnet_embeddings.shape == (len(catalogue_item_ids), 768)
    assert np.isfinite(mpnet_embeddings).all(), "Non-finite embeddings"
    save_embeddings(
        mpnet_output_path, catalogue_item_ids, mpnet_embeddings,
        mpnet, mpnet_model_id, prompt='',
    )
print(mpnet_embeddings.shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/all-mpnet-base-v2.npz
(34220, 768)


### E5-base-v2

- About 110 million parameters, 12 Transformer layers.
- Output dimension: 768.
- No task instruction. Prefix each product text with `query: ` when using embeddings as features.
- English texts only; inputs longer than 512 tokens are truncated. [Model card](https://huggingface.co/intfloat/e5-base-v2).


In [12]:
e5base_model_id = "intfloat/e5-base-v2"
e5base_output_path = embedding_dir / "e5-base-v2.npz"

if e5base_output_path.exists():
    e5base_embeddings = load_embeddings(
        e5base_output_path, catalogue_item_ids, e5base_model_id, 'query: ', 768,
    )
else:
    e5base = SentenceTransformer(
        e5base_model_id,
        device=device,
        cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
    )
    e5base_embeddings = e5base.encode(
        catalogue_texts,
        batch_size=full_batch_size,
        prompt='query: ',
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    assert e5base_embeddings.shape == (len(catalogue_item_ids), 768)
    assert np.isfinite(e5base_embeddings).all(), "Non-finite embeddings"
    save_embeddings(
        e5base_output_path, catalogue_item_ids, e5base_embeddings,
        e5base, e5base_model_id, prompt='query: ',
    )
print(e5base_embeddings.shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/e5-base-v2.npz
(34220, 768)


### Qwen3-Embedding-0.6B

- About 0.6 billion parameters, 28 Transformer layers.
- Output dimension: 1024; input limit: 32,768 tokens.
- Instruction-based. Use `Instruct: {instruction}\nQuery:{text}`.
- Use eager attention on this MPS setup; SDPA returned non-finite vectors. [Model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).


In [13]:
qwen_prompt = f"Instruct: {PILOT_INSTRUCTION}\nQuery:"

print("\nWith instruction:")
print(qwen_prompt + sample_texts[0])



With instruction:
Instruct: Encode this sentence for reccomendation purposes
Query:Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [14]:
qwen_model_id = "Qwen/Qwen3-Embedding-0.6B"
qwen = None
qwen_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": qwen_prompt,
}.items():
    output_path = embedding_dir / f"Qwen3-Embedding-0.6B-{mode}.npz"
    if output_path.exists():
        qwen_embeddings[mode] = load_embeddings(
            output_path, catalogue_item_ids, qwen_model_id, prompt, 1024,
        )
    else:
        if qwen is None:
            qwen = SentenceTransformer(
                qwen_model_id,
                device=device,
                cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
                model_kwargs={"attn_implementation": "eager", "dtype": torch.float32},
                tokenizer_kwargs={"padding_side": "left"},
            )
        qwen_embeddings[mode] = qwen.encode(
            catalogue_texts,
            batch_size=16,
            prompt=prompt,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        assert qwen_embeddings[mode].shape == (len(catalogue_item_ids), 1024)
        assert np.isfinite(qwen_embeddings[mode]).all(), "Non-finite embeddings"
        save_embeddings(
            output_path, catalogue_item_ids, qwen_embeddings[mode],
            qwen, qwen_model_id, prompt=prompt,
        )
    print(mode, qwen_embeddings[mode].shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/Qwen3-Embedding-0.6B-without_instruction.npz
without_instruction (34220, 1024)
Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/Qwen3-Embedding-0.6B-with_instruction.npz
with_instruction (34220, 1024)


### Multilingual-E5-large-instruct

- About 0.6 billion parameters, 24 Transformer layers.
- Output dimension: 1024; input limit: 512 tokens.
- Instruction-based. Use `Instruct: {instruction}\nQuery: {text}`.
- Encode both with and without the shared instruction. [Model card](https://huggingface.co/intfloat/multilingual-e5-large-instruct).


In [15]:
e5instruct_prompt = f"Instruct: {PILOT_INSTRUCTION}\nQuery: "

print("\nWith instruction:")
print(e5instruct_prompt + sample_texts[0])



With instruction:
Instruct: Encode this sentence for reccomendation purposes
Query: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [16]:
e5instruct_model_id = "intfloat/multilingual-e5-large-instruct"
e5instruct = None
e5instruct_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": e5instruct_prompt,
}.items():
    output_path = embedding_dir / f"multilingual-e5-large-instruct-{mode}.npz"
    if output_path.exists():
        e5instruct_embeddings[mode] = load_embeddings(
            output_path, catalogue_item_ids, e5instruct_model_id, prompt, 1024,
        )
    else:
        if e5instruct is None:
            e5instruct = SentenceTransformer(
                e5instruct_model_id,
                device=device,
                cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
            )
        e5instruct_embeddings[mode] = e5instruct.encode(
            catalogue_texts,
            batch_size=full_batch_size,
            prompt=prompt,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        assert e5instruct_embeddings[mode].shape == (len(catalogue_item_ids), 1024)
        assert np.isfinite(e5instruct_embeddings[mode]).all(), "Non-finite embeddings"
        save_embeddings(
            output_path, catalogue_item_ids, e5instruct_embeddings[mode],
            e5instruct, e5instruct_model_id, prompt=prompt,
        )
    print(mode, e5instruct_embeddings[mode].shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/multilingual-e5-large-instruct-without_instruction.npz
without_instruction (34220, 1024)
Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/multilingual-e5-large-instruct-with_instruction.npz
with_instruction (34220, 1024)


### INSTRUCTOR-base

- About 110 million parameters, based on a T5 encoder.
- Output dimension: 768; input limit: 512 tokens.
- Instruction-based. Pass the shared instruction through `prompt=` and exclude its tokens from pooling.
- Encode both with and without the instruction. [Model card](https://huggingface.co/hkunlp/instructor-base).


In [17]:
instructor_prompt = PILOT_INSTRUCTION + " "

print("Without instruction:")
print(sample_texts[0])
print("\nWith instruction:")
print(instructor_prompt + sample_texts[0])


Without instruction:
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

With instruction:
Encode this sentence for reccomendation purposes Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [18]:
instructor_model_id = "hkunlp/instructor-base"
instructor = None
instructor_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": instructor_prompt,
}.items():
    output_path = embedding_dir / f"instructor-base-{mode}.npz"
    if output_path.exists():
        instructor_embeddings[mode] = load_embeddings(
            output_path, catalogue_item_ids, instructor_model_id, prompt, 768,
        )
    else:
        if instructor is None:
            instructor = SentenceTransformer(
                instructor_model_id,
                device=device,
                cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
            )
            instructor.set_pooling_include_prompt(False)
        instructor_embeddings[mode] = instructor.encode(
            catalogue_texts,
            batch_size=full_batch_size,
            prompt=prompt,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        assert instructor_embeddings[mode].shape == (len(catalogue_item_ids), 768)
        assert np.isfinite(instructor_embeddings[mode]).all(), "Non-finite embeddings"
        save_embeddings(
            output_path, catalogue_item_ids, instructor_embeddings[mode],
            instructor, instructor_model_id, prompt=prompt,
        )
    print(mode, instructor_embeddings[mode].shape)


Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/instructor-base-without_instruction.npz
without_instruction (34220, 768)
Loaded 34,220 products from /Users/matouskovar/FIT/NI-PML/artifacts/embeddings/electronics_carec_035_seed2023/instructor-base-with_instruction.npz
with_instruction (34220, 768)


## Embedding shapes

- All nine conditions cover the full catalogue. Run the encoder sections above before showing their output shapes.


In [19]:
pilot_shapes = {
    "MiniLM": minilm_embeddings.shape,
    "MPNet": mpnet_embeddings.shape,
    "E5-base": e5base_embeddings.shape,
}
for model_name, embeddings in {
    "Qwen": qwen_embeddings,
    "E5-instruct": e5instruct_embeddings,
    "INSTRUCTOR": instructor_embeddings,
}.items():
    for mode, vectors in embeddings.items():
        pilot_shapes[f"{model_name}/{mode}"] = vectors.shape

display(pd.Series(pilot_shapes, name="shape"))


MiniLM                              (34220, 384)
MPNet                               (34220, 768)
E5-base                             (34220, 768)
Qwen/without_instruction           (34220, 1024)
Qwen/with_instruction              (34220, 1024)
E5-instruct/without_instruction    (34220, 1024)
E5-instruct/with_instruction       (34220, 1024)
INSTRUCTOR/without_instruction      (34220, 768)
INSTRUCTOR/with_instruction         (34220, 768)
Name: shape, dtype: object